# Correlation, Causation & Confounding

Companion notebook for the [Correlation & Confounding lesson](https://ml-viz-ruby.vercel.app/courses/causal-inference/01-correlation-and-confounding).

We **simulate** data where a treatment has zero true effect but a confounder fakes a strong
correlation — then show that adjusting for the confounder reveals the truth, and reproduce
**Simpson's paradox** where an association reverses under stratification. Pure NumPy.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive.

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

## 1 — A confounder fakes an effect

Health Z drives BOTH who gets treated and who recovers. The treatment itself does nothing (true
effect = 0), yet the naive treated-vs-untreated comparison shows a big 'effect'.

In [ ]:
n = 5000
Z = rng.random(n)                                  # latent health in [0,1]
T = (rng.random(n) < Z).astype(int)                # healthier patients more likely treated
# recovery depends ONLY on health Z, not on treatment T (true effect = 0)
Y = (rng.random(n) < Z).astype(int)

naive = Y[T==1].mean() - Y[T==0].mean()
print(f'true causal effect of treatment: 0.000 (by construction)')
print(f'naive treated - untreated:       {naive:+.3f}  <- looks like the treatment helps a lot!')

## 2 — Adjusting for the confounder reveals the truth

Compare treated vs untreated *within* strata of Z, then average over Z (the backdoor adjustment).
The fake effect collapses to ~0.

In [ ]:
def adjusted_effect(Z, T, Y, bins=10):
    edges = np.linspace(0, 1, bins + 1)
    eff, wts = [], []
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (Z >= lo) & (Z < hi)
        if (T[m]==1).sum() and (T[m]==0).sum():
            eff.append(Y[m & (T==1)].mean() - Y[m & (T==0)].mean())
            wts.append(m.sum())
    return np.average(eff, weights=wts)

print(f'naive effect:               {naive:+.3f}')
print(f'confounder-adjusted effect: {adjusted_effect(Z, T, Y):+.3f}  <- ~0, the truth')

## 3 — Simpson's paradox: the association reverses

We build data where, overall, the treated recover LESS, but within every subgroup they recover MORE
— because the treatment was given more often in the harder subgroup.

In [ ]:
# subgroup A (easy, high base recovery), subgroup B (hard, low base recovery)
def make_group(n, base, treat_rate, boost):
    T = (rng.random(n) < treat_rate).astype(int)
    Y = (rng.random(n) < base + boost * T).astype(int)   # treatment genuinely helps (+boost)
    return T, Y

Ta, Ya = make_group(3000, base=0.80, treat_rate=0.15, boost=0.05)   # easy, rarely treated
Tb, Yb = make_group(3000, base=0.20, treat_rate=0.85, boost=0.05)   # hard, usually treated
T = np.r_[Ta, Tb]; Y = np.r_[Ya, Yb]

print('WITHIN subgroup A: treated - untreated =', round(Ya[Ta==1].mean() - Ya[Ta==0].mean(), 3))
print('WITHIN subgroup B: treated - untreated =', round(Yb[Tb==1].mean() - Yb[Tb==0].mean(), 3))
print('OVERALL (pooled):  treated - untreated =', round(Y[T==1].mean() - Y[T==0].mean(), 3),
      ' <- reversed sign!')

## ✏️ Your turn

**Exercise.** Implement `naive_effect(T, Y)` (treated mean minus untreated mean) and
`stratified_effect(group, T, Y)` — the treatment effect averaged within each subgroup, weighted by
subgroup size. The gap between them is the confounding bias.

In [ ]:
def naive_effect(T, Y):
    # TODO(you): mean outcome of treated minus mean outcome of untreated
    return ...

def stratified_effect(group, T, Y):
    # TODO(you): within-subgroup (treated-untreated) effect, averaged weighted by subgroup size
    return ...

In [ ]:
# This assert cell passes silently when your implementation is correct.
group = np.r_[np.zeros(3000), np.ones(3000)]      # subgroup label A=0, B=1
assert np.isclose(naive_effect(T, Y), Y[T==1].mean() - Y[T==0].mean())
# pooled effect is negative (Simpson) but the stratified effect is positive (the truth)
assert naive_effect(T, Y) < 0
assert stratified_effect(group, T, Y) > 0
print(f'\u2713 naive={naive_effect(T,Y):+.3f} (misleading)  stratified={stratified_effect(group,T,Y):+.3f} (correct)')

<details>
<summary>Solution</summary>

```python
def naive_effect(T, Y):
    return Y[T==1].mean() - Y[T==0].mean()

def stratified_effect(group, T, Y):
    effs, wts = [], []
    for g in np.unique(group):
        m = group == g
        effs.append(Y[m & (T==1)].mean() - Y[m & (T==0)].mean())
        wts.append(m.sum())
    return np.average(effs, weights=wts)
```

The pooled and stratified estimates disagree because the subgroup is a confounder (it affects both
treatment assignment and outcome). Which estimate is *correct* depends on the causal structure —
here the subgroup is a confounder, so the stratified (adjusted) effect is the right one.

</details>